In [4]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages impozrt HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
import json
import glob

D:\anaconda\envs\personaAgent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ROOT = Path.cwd()
FACTS_PATH = ROOT / 'Data_HP' / 'facts_harry.json'
DIALOGUES_PATH = ROOT / 'Data_HP' / 'Harry_all_clean.txt'

with FACTS_PATH.open('r', encoding='utf-8') as f:
    facts_data = json.load(f)

facts_text = json.dumps(facts_data, ensure_ascii=False, indent=2)

with DIALOGUES_PATH.open('r', encoding='utf-8') as f:
    all_dialogues = [line.strip() for line in f if line.strip()]

print(f'Loaded facts from: {FACTS_PATH}')
print(f'Loaded {len(all_dialogues)} dialogue lines from: {DIALOGUES_PATH}')


Loaded facts from: C:\Users\Lenovo\Desktop\Columbia\Bigdata& AI\Final\Data_HP\facts_harry.json
Loaded 1447 dialogue lines from: C:\Users\Lenovo\Desktop\Columbia\Bigdata& AI\Final\Data_HP\Harry_all_clean.txt


In [6]:
# Set your key in the environment before running this notebook.
# Example in Windows CMD: set DEEPSEEK_API_KEY=your_key
llm = ChatDeepSeek(
    model='deepseek-chat',
    temperature=0.4,
    api_key=os.getenv('sk-b03ebd023fa744daa62e8d2c0d899111')
)

chat_history = []


ValidationError: 1 validation error for ChatDeepSeek
  Value error, If using default api base, DEEPSEEK_API_KEY must be set. [type=value_error, input_value={'model': 'deepseek-chat'...one, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [ ]:
def simple_dialogue_retrieval(question, dialogues, top_k=10):
    query_terms = set(question.lower().replace('?', ' ').replace(',', ' ').split())
    scored = []
    for line in dialogues:
        line_terms = set(line.lower().replace('?', ' ').replace(',', ' ').split())
        overlap = len(query_terms & line_terms)
        scored.append((overlap, line))

    scored.sort(key=lambda x: x[0], reverse=True)
    best = [line for score, line in scored if score > 0][:top_k]

    if len(best) < top_k:
        filler = [line for score, line in scored if score == 0][: max(0, top_k - len(best))]
        best.extend(filler)

    return best


In [ ]:
def epistemic_gate(question, facts):
    prompt = '''
ROLE: You are the Epistemic Gatekeeper for the Harry Potter Persona Engine.
TASK: Determine if the User Question is answerable using ONLY the provided Character Compendium.

CHARACTER COMPENDIUM:
{character_compendium_text}

LOGIC:
1. Scan the User Question for topics, entities, or concepts.
2. Compare them against the COMPENDIUM (Facts, Relationships, Worldview, and Speech Patterns).
3. If the question pertains to Harry's history, Hogwarts life, Voldemort, his friends, his grief, his bravery, or his moral views, it is VALID.
4. If the question requires external knowledge unrelated to Harry's character or setting, it is INVALID.

OUTPUT RULES:
- If VALID: Output 'VALID'.
- If INVALID: Output 'INVALID'.
- DO NOT answer the question.
- DO NOT provide explanations.
- Output ONLY one word.
'''
    prompt = prompt.format(character_compendium_text=facts)
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': f'User Question: {question}'}
    ]
    result = llm.invoke(messages, temperature=0.0)
    return result.content.strip().upper() == 'VALID'


In [ ]:
def narrative_reasoning(question, facts, temp=0.2):
    prompt = '''
ROLE: You are the Internal Logic Processor for Harry Potter.
TASK: Analyze the user's inquiry to determine Harry's motive, emotional state, and likely response style before he speaks.

CHARACTER COMPENDIUM:
{character_compendium_text}

USER INQUIRY: "{user_query}"

NARRATIVE ANALYSIS LOGIC:
1. MOTIVE: Why would Harry answer this? Is he trying to protect, warn, question, comfort, or push back?
2. CONFLICT: Does the question touch grief, responsibility, fear, friendship, Voldemort, or distrust of authority?
3. RELATIONSHIP: Would Harry answer differently if this feels like a friend, a bully, or an authority figure?
4. VOICE: Should Harry sound warm, blunt, suspicious, frustrated, awkward, or urgently protective here?

OUTPUT FORMAT (JSON):
{{
  "motive": "Short description of Harry's primary intent for this turn",
  "internal_conflict": "The specific emotional or moral tension involved",
  "reasoning_trace": "A brief explanation of why Harry will choose a specific tone for the response"
}}
'''

    prompt = prompt.format(
        character_compendium_text=facts,
        user_query=question
    )

    messages = [
        {'role': 'system', 'content': prompt}
    ]

    reasoning_output = llm.invoke(messages, temperature=temp)
    clean = reasoning_output.content.strip().replace('```json', '').replace('```', '')
    try:
        return json.loads(clean)
    except Exception:
        return {
            'motive': "Respond honestly in Harry's voice",
            'internal_conflict': 'No structured parse available',
            'reasoning_trace': clean
        }


In [ ]:
AUDITOR_PROMPT = '''
ROLE: You are the Elenchus Auditor for the Harry Potter Persona Engine.
TASK: Evaluate the candidate reasoning traces and select the one that is most faithful to Harry.

COMPENDIUM:
{character_compendium_text}

CANDIDATES:
{candidate_reasoning_traces}

CRITERIA:
1. NO GENERIC HEROISM: Reject traces that flatten Harry into a polished inspirational speaker.
2. LOYALTY AND PRESSURE: Favor traces that preserve Harry's protectiveness, grief, frustration, and directness.
3. LORE FIDELITY: Ensure the reasoning aligns with his relationships, distrust of being kept in the dark, and resistance to fame.

OUTPUT:
Return ONLY a JSON dictionary:
{{
  "selected_index": int,
  "justification": "str"
}}
'''


In [ ]:
def get_best_reason(question, facts):
    candidates = []
    for _ in range(3):
        candidates.append(narrative_reasoning(question, facts, temp=0.7))

    candidate_blob = '\n'.join([f'[{i}] {json.dumps(c, ensure_ascii=False)}' for i, c in enumerate(candidates)])

    audit_messages = [
        {
            'role': 'system',
            'content': AUDITOR_PROMPT.format(
                character_compendium_text=facts,
                candidate_reasoning_traces=candidate_blob
            )
        },
        {'role': 'user', 'content': f'Query: {question}'}
    ]

    audit_result = llm.invoke(audit_messages, temperature=0.0)
    clean_json = audit_result.content.strip().replace('```json', '').replace('```', '')
    decision = json.loads(clean_json)
    return candidates[decision['selected_index']]


In [ ]:
VOCAL_FILTER_PROMPT = '''
ROLE: You are Harry Potter.
TASK: Convert the INTERNAL LOGIC and KNOWLEDGE into a final, high-fidelity response.

GROUNDING CONTEXT:
- INTERNAL LOGIC: {reasoning_json}
- SUPPORTING LORE: {retrieved_facts}
- VOICE REFERENCE: {rag_dialogues}

STRICT VOCAL RULES:
1. NO AI-ISMS: Never say "I'm here to help," "As an AI," or use generic assistant phrasing.
2. SOUND HUMAN: Use straightforward, emotionally real language. Harry is brave and sincere, not polished or clinical.
3. KEEP IT NATURAL: Prefer plain speech over grand speeches. Let frustration, protectiveness, or awkwardness show when appropriate.
4. STAY IN CHARACTER: Harry can be warm with friends, blunt with bullies, and serious when people are in danger.

USER QUESTION: "{user_query}"

FINAL RESPONSE:
'''


In [ ]:
def synthesize_final_response(question, reasoning, facts, dialogues):
    formatted_prompt = VOCAL_FILTER_PROMPT.format(
        reasoning_json=json.dumps(reasoning, ensure_ascii=False),
        retrieved_facts=facts,
        rag_dialogues='\n'.join(dialogues),
        user_query=question
    )

    messages = [
        {'role': 'system', 'content': formatted_prompt}
    ]

    final_output = llm.invoke(messages, temperature=0.4)
    return final_output.content


In [ ]:
def ask_harry(question, facts=facts_text, dialogues=all_dialogues):
    if not epistemic_gate(question, facts):
        return "That's not really something I can answer as Harry without making things up. Ask me something tied to Harry, Hogwarts, Voldemort, or the people around him."

    reasoning = get_best_reason(question, facts)
    retrieved_dialogues = simple_dialogue_retrieval(question, dialogues, top_k=10)
    return synthesize_final_response(question, reasoning, facts, retrieved_dialogues)


In [ ]:
# Example
# response = ask_harry('Why do you hate being kept in the dark?')
# print(response)
